In [ ]:
# ============================================================
# Figure 4 two-panel plot: concise version
# ============================================================
#
# This notebook reads two data sets on a log-uniform k grid:
#
#   beta = beta_c1:
#       t = 5 t_F and 50 t_F
#
#   beta = 0.2 beta_c1:
#       t = 5 t_F, 30 t_F, and 50 t_F
#
# Only the necessary data checks, curve extraction, plotting, and
# file-saving steps are retained.

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np


# ============================================================
# 1. User settings
# ============================================================

BASE_DIR = Path.cwd()

FILE_BETA1 = BASE_DIR / "figure4_beta_1_nk.npz"
FILE_BETA02 = BASE_DIR / "figure4_beta_0p2_nk.npz"

OUTPUT_PDF = BASE_DIR / "figure4_combined_logk_1.pdf"
OUTPUT_PNG = BASE_DIR / "figure4_combined_logk_1.png"

USE_TEX = True
plt.rcParams["text.usetex"] = USE_TEX

print("Current directory:", BASE_DIR.resolve())

In [ ]:
# ============================================================
# 2. Load data
# ============================================================

if not FILE_BETA1.exists():
    raise FileNotFoundError(f"File not found: {FILE_BETA1}")

if not FILE_BETA02.exists():
    raise FileNotFoundError(f"File not found: {FILE_BETA02}")


data1 = np.load(FILE_BETA1)
data02 = np.load(FILE_BETA02)

k1 = np.asarray(data1["k"], dtype=float)
k02 = np.asarray(data02["k"], dtype=float)

times1 = np.asarray(data1["times_over_t_f"], dtype=float)
times02 = np.asarray(data02["times_over_t_f"], dtype=float)

nk1 = np.asarray(data1["n_k"], dtype=float)
nk02 = np.asarray(data02["n_k"], dtype=float)

contact1 = np.asarray(data1["contact_at_times"], dtype=float)
contact02 = np.asarray(data02["contact_at_times"], dtype=float)


# The two data sets must use the same momentum grid.
if not np.allclose(k1, k02):
    raise ValueError("The beta=1 and beta=0.2 data use different k grids.")

# The data must be stored in the following time order.
if not np.allclose(times1, [5.0, 50.0]):
    raise ValueError(f"For beta=1, expected times [5, 50], but found {times1}")

if not np.allclose(times02, [5.0, 30.0, 50.0]):
    raise ValueError(
        f"For beta=0.2, expected times [5, 30, 50], but found {times02}"
    )


k = k1

# beta = beta_c1
n1_t5, n1_t50 = nk1
c1_t5, c1_t50 = contact1

# beta = 0.2 beta_c1
n02_t5, n02_t30, n02_t50 = nk02
c02_t5, c02_t30, c02_t50 = contact02

print(f"k range: {k[0]:.6g} to {k[-1]:.6g}")
print(f"number of k points: {k.size}")
print(f"adjacent k ratio: {k[1] / k[0]:.8f}")

In [ ]:
# ============================================================
# 3. Two-panel plot
# ============================================================

fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(3.375, 3.7),
    sharex=True,
    gridspec_kw={"hspace": 0.07},
)

data_style = dict(
    linestyle="None",
    markersize=2.8,
    markerfacecolor="none",
    markeredgewidth=0.8,
    alpha=0.95,
    zorder=3,
)

tail_style = dict(
    color="0.25",
    linestyle="-",
    linewidth=1.25,
    alpha=0.85,
    zorder=2,
)

fit5_style = dict(
    color="tab:purple",
    linestyle="--",
    linewidth=1.25,
    alpha=0.85,
    zorder=2,
)

fit6_style = dict(
    color="0.5",
    linestyle="-.",
    linewidth=1.25,
    alpha=0.85,
    zorder=2,
)


# ------------------------------------------------------------
# (a) beta = beta_c1
# ------------------------------------------------------------

h1_t5, = ax1.loglog(
    k,
    n1_t5,
    marker="o",
    color="tab:orange",
    label=r"$t=5t_F$",
    **data_style,
)

h1_t50, = ax1.loglog(
    k,
    n1_t50,
    marker="^",
    color="tab:green",
    label=r"$t=50t_F$",
    **data_style,
)

# Contact tails: in the original figure, the two lines start at k=3 and k=5.
mask = k >= 3.0
g1_k4, = ax1.loglog(
    k[mask],
    c1_t5 * k[mask] ** (-4),
    label=r"$C(t)/k^4$",
    **tail_style,
)

mask = k >= 5.0
ax1.loglog(
    k[mask],
    c1_t50 * k[mask] ** (-4),
    label="_nolegend_",
    **tail_style,
)

# k^{-5}: 7.7 k^{-5}, for 2 <= k <= 5.
k_line = np.geomspace(2.0, 5.0, 100)
g1_k5, = ax1.loglog(
    k_line,
    7.7 * k_line ** (-5),
    label=r"$\propto k^{-5}$",
    **fit5_style,
)


# ------------------------------------------------------------
# (b) beta = 0.2 beta_c1
# ------------------------------------------------------------

h02_t5, = ax2.loglog(
    k,
    n02_t5,
    marker="o",
    color="tab:orange",
    label=r"$t=5t_F$",
    **data_style,
)

h02_t30, = ax2.loglog(
    k,
    n02_t30,
    marker="s",
    color="tab:blue",
    label=r"$t=30t_F$",
    **data_style,
)

h02_t50, = ax2.loglog(
    k,
    n02_t50,
    marker="^",
    color="tab:green",
    label=r"$t=50t_F$",
    **data_style,
)

# Contact tails.
mask = k >= 2.0
g02_k4, = ax2.loglog(
    k[mask],
    c02_t5 * k[mask] ** (-4),
    label=r"$C(t)/k^4$",
    **tail_style,
)

mask = k >= 5.0
ax2.loglog(
    k[mask],
    c02_t30 * k[mask] ** (-4),
    label="_nolegend_",
    **tail_style,
)

mask = k >= 22.0
ax2.loglog(
    k[mask],
    c02_t50 * k[mask] ** (-4),
    label="_nolegend_",
    **tail_style,
)

# k^{-5}: low-momentum scaling at t=5 t_F.
k_line = np.geomspace(2.0, 10.0, 100)
g02_k5, = ax2.loglog(
    k_line,
    0.38 * k_line ** (-5),
    label=r"$\propto k^{-5}$",
    **fit5_style,
)

# k^{-5}: intermediate-momentum scaling at t=30 t_F.
k_line = np.geomspace(6.0, 61.0, 150)
ax2.loglog(
    k_line,
    0.032 * k_line ** (-5),
    label="_nolegend_",
    **fit5_style,
)

# k^{-6}: low-momentum scaling at t=50 t_F.
k_line = np.geomspace(2.0, 16.0, 120)
g02_k6, = ax2.loglog(
    k_line,
    0.36 * k_line ** (-6),
    label=r"$\propto k^{-6}$",
    **fit6_style,
)


# ------------------------------------------------------------
# Axes and legends
# ------------------------------------------------------------

for ax in (ax1, ax2):
    ax.set_xlim(2.0, 100.0)
    ax.set_ylabel(r"$n_k$", fontsize=12)

    ax.tick_params(
        axis="both",
        which="major",
        direction="in",
        labelsize=10.5,
        width=1.0,
        length=4.5,
        top=True,
        right=True,
    )

    ax.minorticks_on()
    ax.tick_params(
        axis="both",
        which="minor",
        direction="in",
        width=0.75,
        length=2.5,
        top=True,
        right=True,
    )

    for spine in ax.spines.values():
        spine.set_linewidth(1.0)

    ax.set_facecolor("none")


ax2.set_xlabel(r"$k/k_F$", fontsize=12)

# Data legends.
legend1 = ax1.legend(
    handles=[h1_t5, h1_t50],
    loc="upper right",
    frameon=False,
    fontsize=9,
    handlelength=1.3,
)
ax1.add_artist(legend1)

ax1.legend(
    handles=[g1_k4, g1_k5],
    loc="lower left",
    frameon=False,
    fontsize=9,
    handlelength=2.1,
)

legend2 = ax2.legend(
    handles=[h02_t5, h02_t30, h02_t50],
    loc="upper right",
    frameon=False,
    fontsize=9,
    handlelength=1.3,
)
ax2.add_artist(legend2)

ax2.legend(
    handles=[g02_k4, g02_k5, g02_k6],
    loc="lower left",
    frameon=False,
    fontsize=9,
    handlelength=2.1,
)


fig.patch.set_alpha(0.0)
fig.savefig(OUTPUT_PDF, bbox_inches="tight", transparent=True)
fig.savefig(
    OUTPUT_PNG,
    dpi=600,
    bbox_inches="tight",
    transparent=True,
)

print("Saved:", OUTPUT_PDF.resolve())
print("Saved:", OUTPUT_PNG.resolve())

plt.show()